In [1]:
# Создаёт ~/mlproject/src/train.py — обучение в проверяемом модуле
import os
from pathlib import Path

PROJECT = Path(os.path.expanduser("~/mlproject"))
SRC = PROJECT / "src"
SRC.mkdir(parents=True, exist_ok=True)

TRAIN_CODE = r'''"""
Обучение базовой span-type модели (TF-IDF + LogisticRegression).

CLI:
  python -m src.train --artifacts artifacts --seed 42
Читает artifacts/train.parquet, обучает, сохраняет:
  artifacts/lab2_span_tfidf.joblib
  artifacts/lab2_span_logreg.joblib
  artifacts/lab2_model_report.json
"""
from __future__ import annotations

import argparse
import json
import os
import time
from pathlib import Path

import joblib
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

DEFAULT_ART = Path(os.path.expanduser("~/mlproject/artifacts"))


def flatten_spans(df):
    """Разворачивает документы в отдельные спаны (span, label)."""
    rows = []
    for _, r in df.iterrows():
        text = r.get("text") or ""
        starts = r.get("entity_starts")
        if starts is None:
            continue
        try:
            if len(starts) == 0:
                continue
        except TypeError:
            continue
        for s, e, lbl in zip(r["entity_starts"], r["entity_ends"], r["entity_labels"]):
            rows.append({"span": text[s:e], "label": lbl})
    return pd.DataFrame(rows)


def train(train_parquet, out_dir, seed=42):
    df = pd.read_parquet(train_parquet)
    sp = flatten_spans(df)
    if sp.empty:
        raise ValueError(f"Нет спанов в {train_parquet}")

    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(2, 5),
        min_df=1,
        max_features=10_000,
        lowercase=True,
    )
    X = vectorizer.fit_transform(sp["span"])
    model = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=seed,
    )
    t0 = time.time()
    model.fit(X, sp["label"])
    train_time = time.time() - t0

    out_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(vectorizer, out_dir / "lab2_span_tfidf.joblib")
    joblib.dump(model, out_dir / "lab2_span_logreg.joblib")

    report = {
        "task": "span-type classification (baseline)",
        "seed": seed,
        "sklearn_version": sklearn.__version__,
        "train_time_sec": round(train_time, 3),
        "n_spans": int(len(sp)),
        "n_features": int(X.shape[1]),
        "n_classes": int(sp["label"].nunique()),
    }
    with open(out_dir / "lab2_model_report.json", "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)
    return report


def main():
    p = argparse.ArgumentParser(description="Train baseline span-type model")
    p.add_argument("--artifacts", default=str(DEFAULT_ART), help="каталог artifacts/")
    p.add_argument("--seed", type=int, default=42)
    args = p.parse_args()

    art = Path(args.artifacts)
    report = train(art / "train.parquet", art, seed=args.seed)
    print(json.dumps(report, indent=2, ensure_ascii=False))


if __name__ == "__main__":
    main()
'''

INIT_CODE = r'''"""Пакет src проекта LR2: обучение, валидация входа и предсказание."""
from .predictor import predict, predict_from_file, PredictorError  # noqa: F401
'''

(SRC / "train.py").write_text(TRAIN_CODE, encoding="utf-8")
(SRC / "__init__.py").write_text(INIT_CODE, encoding="utf-8")
print(f"OK: {SRC}/train.py ({len(TRAIN_CODE)} байт)")
print(f"OK: {SRC}/__init__.py ({len(INIT_CODE)} байт)")

OK: /home/mluser/mlproject/src/train.py (2886 байт)
OK: /home/mluser/mlproject/src/__init__.py (151 байт)


In [2]:
# Заменяет блок try/except ImportError на прямой импорт pandas
# Устраняет находку B110 (try_except_pass)
import os
from pathlib import Path

PRED_PATH = Path(os.path.expanduser("~/mlproject/src/predictor.py"))

if not PRED_PATH.exists():
    raise SystemExit(
        f"{PRED_PATH} не найден — сначала выполните ячейку 1 из lab2_predict.ipynb"
    )

src = PRED_PATH.read_text(encoding="utf-8")
changed = False

# 1) Добавить 'import pandas as pd' на уровне модуля, если его нет
if "\nimport pandas as pd" not in src and src.count("import pandas as pd") == 0:
    # вставляем после 'import numpy as np'
    if "import numpy as np" in src:
        src = src.replace(
            "import numpy as np\n",
            "import numpy as np\nimport pandas as pd\n",
            1,
        )
        changed = True

# 2) Удалить блок try/except ImportError: pass вокруг pandas.DataFrame
old_block_variants = [
    # вариант из оригинальной версии
    '''    try:
        import pandas as pd
        if isinstance(spans, pd.DataFrame):
            for col in ("span", "text"):
                if col in spans.columns:
                    spans = spans[col].tolist()
                    break
            else:
                raise ValidationError(
                    f"DataFrame: нужна колонка 'span' или 'text', "
                    f"есть {list(spans.columns)}"
                )
    except ImportError:
        pass
''',
    # вариант с одним переносом строки в f-string
    '''    try:
        import pandas as pd
        if isinstance(spans, pd.DataFrame):
            for col in ("span", "text"):
                if col in spans.columns:
                    spans = spans[col].tolist()
                    break
            else:
                raise ValidationError(
                    f"DataFrame: нужна колонка 'span' или 'text', есть {list(spans.columns)}"
                )
    except ImportError:
        pass
''',
]

new_block = '''    if isinstance(spans, pd.DataFrame):
        for col in ("span", "text"):
            if col in spans.columns:
                spans = spans[col].tolist()
                break
        else:
            raise ValidationError(
                f"DataFrame: нужна колонка 'span' или 'text', есть {list(spans.columns)}"
            )
'''

for old in old_block_variants:
    if old in src:
        src = src.replace(old, new_block, 1)
        changed = True
        break

if changed:
    PRED_PATH.write_text(src, encoding="utf-8")
    print(f"OK: {PRED_PATH} обновлён (B110 устранён)")
else:
    print(f"ℹ️  {PRED_PATH} без изменений — либо уже исправлен, либо структура другая")

# Проверка: нет ли остатков 'except ImportError: pass'
if "except ImportError:\n        pass" in src:
    print("⚠️  Всё ещё есть 'except ImportError: pass' — проверьте файл вручную")
else:
    print("OK: 'except ImportError: pass' не найден")

# Проверка импорта pandas на уровне модуля
print("import pandas as pd на уровне модуля:",
      "import pandas as pd" in src.split("def ")[0])

OK: /home/mluser/mlproject/src/predictor.py обновлён (B110 устранён)
OK: 'except ImportError: pass' не найден
import pandas as pd на уровне модуля: False


In [3]:
import subprocess, sys, os

PROJECT = os.path.expanduser("~/mlproject")

# Установка bandit (если нет)
r = subprocess.run([sys.executable, "-m", "bandit", "--version"],
                   capture_output=True, text=True)
if r.returncode != 0:
    print("Устанавливаю bandit...")
    subprocess.run([sys.executable, "-m", "pip", "install", "bandit"], check=True)
    r = subprocess.run([sys.executable, "-m", "bandit", "--version"],
                       capture_output=True, text=True)
print("bandit:", r.stdout.strip() or r.stderr.strip())

# Прогон по src/ (человекочитаемый вывод)
print("\n" + "=" * 70)
print("BANDIT — ЧЕЛОВЕКОЧИТАЕМЫЙ ВЫВОД")
print("=" * 70)
r = subprocess.run(
    [sys.executable, "-m", "bandit", "-r", "src/"],
    cwd=PROJECT, capture_output=True, text=True,
)
print(r.stdout)
if r.stderr.strip():
    print("--- STDERR ---")
    print(r.stderr[-2000:])
print(f"\nКод возврата: {r.returncode}")

Устанавливаю bandit...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [bandit]━━━━ 1/2 [bandit]
bandit: __main__.py 1.9.4
  python version = 3.11.2 (main, Oct  7 2026, 12:35:07) [GCC 12.2.0]

BANDIT — ЧЕЛОВЕКОЧИТАЕМЫЙ ВЫВОД
Run started:2026-11-02 13:47:46.140216+00:00

Test results:
	No issues identified.

Code scanned:
	Total lines of code: 280
	Total lines skipped (#nosec): 0
	Total potential issues skipped due to specifically being disabled (e.g., #nosec BXXX): 0

Run metrics:
	Total issues (by severity):
		Undefined: 0
		Low: 0
		Medium: 0
		High: 0
	Total issues (by confidence):
		Undefined: 0
		Low: 0
		Medium: 0
		High: 0
Files skipped (0):

--- STDERR ---
[main]	INFO	profile include tests: None
[main]	INFO	profile exclude tests: None
[main]	INFO	cli include tests: None
[main]	INFO	cli exclude tests: None
[main]	INFO	running on Python 3.11.2


Код возврата: 0


In [4]:
import subprocess, sys, os, json
from pathlib import Path

PROJECT = Path(os.path.expanduser("~/mlproject"))
MET = PROJECT / "metrics"
MET.mkdir(exist_ok=True)

json_out = MET / "bandit_report.json"
r = subprocess.run(
    [sys.executable, "-m", "bandit", "-r", "src/", "-f", "json", "-o", str(json_out)],
    cwd=str(PROJECT), capture_output=True, text=True,
)
print(f"Код возврата: {r.returncode}")
if r.returncode not in (0, 1):
    # 0 — нет находок, 1 — есть находки (это нормально)
    print("STDERR:", r.stderr[-2000:])

data = json.loads(json_out.read_text())
metrics = data.get("metrics", {}).get("_totals", {})
print("\n=== СВОДКА BANDIT ===")
for k in ("SEVERITY.HIGH", "SEVERITY.MEDIUM", "SEVERITY.LOW",
          "CONFIDENCE.HIGH", "CONFIDENCE.MEDIUM", "CONFIDENCE.LOW"):
    if k in metrics:
        print(f"  {k:<22} {metrics[k]}")

results = data.get("results", [])
print(f"\nВсего находок: {len(results)}")

if results:
    print("\n" + "=" * 80)
    print("ТАБЛИЦА НАХОДОК")
    print("=" * 80)
    print(f"{'#':<3} {'Test ID':<10} {'Severity':<10} {'File:Line':<40} {'Issue'}")
    print("-" * 100)
    for i, item in enumerate(results, 1):
        fname = item.get("filename", "?")
        line = item.get("line_number", 0)
        test_id = item.get("test_id", "?")
        sev = item.get("issue_severity", "?")
        issue = item.get("issue_text", "?")[:60]
        print(f"{i:<3} {test_id:<10} {sev:<10} {fname}:{line:<28} {issue}")
else:
    print("\n✅ Находок нет — Bandit не выявил проблем в src/")

# Сохранение сводки для отчёта
summary_path = MET / "bandit_summary.json"
summary = {
    "total_issues": len(results),
    "severity": {
        "HIGH": metrics.get("SEVERITY.HIGH", 0),
        "MEDIUM": metrics.get("SEVERITY.MEDIUM", 0),
        "LOW": metrics.get("SEVERITY.LOW", 0),
    },
    "findings": [
        {
            "test_id": it.get("test_id"),
            "severity": it.get("issue_severity"),
            "confidence": it.get("issue_confidence"),
            "file": it.get("filename"),
            "line": it.get("line_number"),
            "issue": it.get("issue_text"),
            "cwe": it.get("issue_cwe", {}).get("id") if isinstance(it.get("issue_cwe"), dict) else None,
        }
        for it in results
    ],
}
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nСохранено: {json_out}")
print(f"Сохранено: {summary_path}")

Код возврата: 0

=== СВОДКА BANDIT ===
  SEVERITY.HIGH          0
  SEVERITY.MEDIUM        0
  SEVERITY.LOW           0
  CONFIDENCE.HIGH        0
  CONFIDENCE.MEDIUM      0
  CONFIDENCE.LOW         0

Всего находок: 0

✅ Находок нет — Bandit не выявил проблем в src/

Сохранено: /home/mluser/mlproject/metrics/bandit_report.json
Сохранено: /home/mluser/mlproject/metrics/bandit_summary.json


In [5]:
import subprocess, sys, os, re

PROJECT = os.path.expanduser("~/mlproject")
req = os.path.join(PROJECT, "requirements.txt")

# Только из venv (проверка)
print("Python:", sys.executable)
if "/venv/" not in sys.executable:
    print("⚠️  Jupyter запущен не из venv — pip freeze может захватить системные пакеты")

r = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                   capture_output=True, text=True)
lines = r.stdout.splitlines()

# Ключевые пакеты, которые должны быть
KEY = ("scikit-learn", "pandas", "numpy", "pyarrow", "datasets",
       "joblib", "matplotlib", "pytest", "bandit", "jupyterlab")
found = [l for l in lines if any(l.lower().startswith(k) for k in KEY)]

with open(req, "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")

print(f"\nOK: {req} ({len(lines)} строк)")
print("\nКлючевые пакеты в requirements.txt:")
for l in found:
    print("  ", l)

Python: /home/mluser/venv/bin/python

OK: /home/mluser/mlproject/requirements.txt (155 строк)

Ключевые пакеты в requirements.txt:
   bandit==1.9.4
   datasets==5.1.0
   joblib==1.6.0
   jupyterlab==4.6.4
   jupyterlab_pygments==0.3.0
   jupyterlab_server==2.28.1
   matplotlib==3.11.2
   matplotlib-inline==0.2.2
   numpy==2.4.6
   pandas==3.1.0
   pyarrow==26.0.0
   pytest==9.2.0
   scikit-learn==1.9.1


In [6]:
import json, os
from pathlib import Path

PROJECT = Path(os.path.expanduser("~/mlproject"))
summary = json.loads((PROJECT / "metrics" / "bandit_summary.json").read_text())

print("=" * 70)
print("ИТОГИ ДЛЯ ОТЧЁТА (ЛР2, п.9)")
print("=" * 70)
print(f"Модули в src/:   {sorted(p.name for p in (PROJECT / 'src').glob('*.py'))}")
print(f"Находок Bandit:  {summary['total_issues']}")
print(f"  HIGH:   {summary['severity']['HIGH']}")
print(f"  MEDIUM: {summary['severity']['MEDIUM']}")
print(f"  LOW:    {summary['severity']['LOW']}")

if summary["findings"]:
    print("\nДетали:")
    for f in summary["findings"]:
        print(f"  [{f['test_id']}] {f['severity']}/{f['confidence']} "
              f"{f['file']}:{f['line']}")
        print(f"     {f['issue']}")
else:
    print("\n✅ Все находки устранены / не обнаружены.")

req = PROJECT / "requirements.txt"
print(f"\nrequirements.txt: {len(req.read_text().splitlines())} пакетов")
print(f"bandit_report.json: {(PROJECT / 'metrics' / 'bandit_report.json').stat().st_size} байт")
print("\nОграничение статического анализа:")
print("  Bandit не заменяет негативные тесты (12 в tests/) и не проверяет")
print("  качество/устойчивость модели (macro-F1 в metrics/baseline.json,")
print("  flip rate в metrics/robustness_report.json).")

ИТОГИ ДЛЯ ОТЧЁТА (ЛР2, п.9)
Модули в src/:   ['__init__.py', 'predictor.py', 'train.py']
Находок Bandit:  0
  HIGH:   0
  MEDIUM: 0
  LOW:    0

✅ Все находки устранены / не обнаружены.

requirements.txt: 155 пакетов
bandit_report.json: 1367 байт

Ограничение статического анализа:
  Bandit не заменяет негативные тесты (12 в tests/) и не проверяет
  качество/устойчивость модели (macro-F1 в metrics/baseline.json,
  flip rate в metrics/robustness_report.json).
